# 16 — The capstone synthesis

Close the capstone by holding two truths at once: across the last notebook you *demonstrated*, with a control, that a quantum-optimal-transport coupling measure can beat the phase-locking value — and here you give that result its honest accounting, the full reckoning of when it helps, at what cost, and where its real value lies.

**Prerequisites:** `11_kuramoto_dyad_and_ground_truth`, `12_naive_phase_embedding_tracks_K`, `13_the_redundancy_reveal`, `14_richer_embedding_applied`, `15_discriminating_experiment`.

**What you'll be able to do**
- Retell the capstone as one connected argument — the redundancy reveal (`13`), the richer embedding (`14`), and the discriminating experiment (`15`) — and say in one sentence what each step settled.
- State the demonstrated result precisely: a controlled, significant case where a richer-embedding QOT measure separates two ensembles the phase-locking value cannot.
- Give the **honest accounting** — *when* QOT adds something over PLV, *at what statistical cost*, and the caveat that a bespoke classical higher-order statistic could also see the same difference.
- Name the genuine value proposition: a single transport / information-geometric framework that reaches higher-order coupling structure through the embedding choice rather than through a hand-picked statistic.
- Close the **coupling-measure row** of the classical↔quantum dictionary the whole course has been assembling.

In `15` you collected on the promise the capstone was built around: two systems with identical PLV, parted by a quantum measure that read the second circular moment PLV is blind to — real, not noise, clear of its null across many seeds. That was the payoff. This notebook is the part a careful demonstration always owes you next: not another experiment, but the mature reading of the one you ran. We celebrate exactly what was shown, no more — and lay the caveats beside it plainly, because a result you can trust is one whose limits you can also state.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qot_course import viz
from qot_course.colors import COLORS

np.random.seed(42)
viz.use_course_style()

## 1. The A→B→C story, in one breath

The capstone asked a single question and refused to answer it cheaply: can a quantum-optimal-transport coupling measure do something the phase-locking value cannot? Notebooks `11` and `12` built the honest bench — a Kuramoto dyad whose coupling $K$ we inject ourselves (so we hold the ground truth), the two classical baselines every measure must clear, and a first quantum pipeline that, reassuringly, tracked $K$ as the baselines did. Then the argument turned in three decisive moves.

**The redundancy reveal — `13` [C].** Rising with $K$ is necessary but not sufficient: PLV rises with $K$ too. So we sharpened the question into a rank correlation and got a clean, uncomfortable result — on the **naive phase qubit**, $\mathrm{Spearman}(\mathrm{PLV}, \mathrm{QMI}) \approx 1$ and $\mathrm{Spearman}(\mathrm{PLV}, \mathrm{Bures}) \approx 1$. The quantum measures were PLV in disguise. And we proved *why* from the operator itself: on that embedding the diagonal of $\rho_{AB}$ is frozen at $\tfrac14 I_4$, the marginals at $I/2$, and the lone coupling-bearing element is the off-block coherence with $|\rho_{AB}[1,2]| = \mathrm{PLV}/4$. A state whose only moving part *is* PLV can feed a measure nothing else. Crucially, this is a property of the **embedding**, not of quantum OT — which told us exactly what to change.

**The richer embedding — `14` [B].** We swapped the two-level qubit for the multi-frequency **qutrit** $\tfrac{1}{\sqrt 3}\bigl(|0\rangle + e^{i\theta}|1\rangle + e^{2i\theta}|2\rangle\bigr)$. Its $9\times 9$ joint density carries the coupling in **two** coherences: $\rho_{AB}[3,1]$, the first circular moment (PLV's channel), and $\rho_{AB}[6,2]$, the **second** moment — a channel that lives on the $|2\rangle$ level the qubit did not possess. We checked it carries real structure (it grows as the dyad locks) and, with a matched-PLV pair, that it is an *independent* degree of freedom: two couplings can share a PLV yet differ in the second moment. The redundancy was broken in principle.

**The discriminating experiment — `15` [A].** We turned that structural opening into a measured advantage. We built two ensembles with identical PLV by construction (same first moment $a_1 = 0.4$) but different second moment ($a_2 = 0.0$ versus $0.3$). PLV was matched to the Monte-Carlo floor; the **naive** embedding's QMI was matched too (exactly as `13` predicted); and the **richer** embedding's QMI *separated* them — by a gap several times the floor, standing clear of an $a_2$-matched null across many seeds. That is the headline the restructure exists to deliver: **a controlled, significant case where QOT beats PLV.**

Let us put the three gaps from `15` side by side, so the whole arc reads in one figure before we account for it.

In [ ]:
# Re-cite the headline numbers from notebook 15 (the discriminating experiment).
# These are the measured gaps between the two matched-PLV ensembles (a1 = 0.4;
# a2 = 0.0 vs 0.3), at N = 150,000 samples per channel, seed = 0 -- reproduced in 15.
# We do not re-run the expensive sweep here: 15 owns the experiment and its controls;
# the synthesis recalls its result to read it.
headline = {
    "PLV": 0.0017,                  # first-moment statistic -- matched by construction
    "naive-embedding QMI": 0.0004,  # naive qubit -> PLV in disguise (notebook 13)
    "richer-embedding QMI": 0.0044, # qutrit reads the 2nd moment -> the separation
}
null_ceiling = 0.0018  # largest a2-matched null gap over 12 seeds (the floor, 15 sec.5)

labels = list(headline)
gaps = np.array([headline[k] for k in labels])
# Grey for the two blind first-moment views; rose (the punchline colour) for the win.
bar_colors = [COLORS["muted"], COLORS["muted"], COLORS["highlight"]]

fig, (ax_bar, ax_schema) = plt.subplots(1, 2, figsize=(12, 5))

# (a) The three gaps, with the noise floor drawn in.
y = np.arange(len(labels))[::-1]
ax_bar.barh(y, gaps, color=bar_colors, edgecolor="white", linewidth=1.2, zorder=3)
ax_bar.axvline(null_ceiling, color=COLORS["text"], lw=1.4, ls="--", zorder=2,
               label="matched-$a_2$ null ceiling\n(Monte-Carlo floor)")
for yi, g in zip(y, gaps):
    ax_bar.annotate(f"{g:.4f}", (g, yi), xytext=(6, 0), textcoords="offset points",
                    va="center", ha="left", fontsize=10, color=COLORS["text"])
ax_bar.set_yticks(y)
ax_bar.set_yticklabels(labels)
ax_bar.set_xlabel("gap between the two matched-PLV ensembles  (nats)")
ax_bar.set_xlim(0, max(gaps) * 1.35)
ax_bar.set_title("Only the richer embedding separates them", pad=10)
ax_bar.legend(loc="lower right", fontsize=9)
ax_bar.grid(axis="y", visible=False)

# (b) What each embedding's rho_AB can see, as a schematic.
ax_schema.set_xlim(0, 1)
ax_schema.set_ylim(0, 1)
ax_schema.axis("off")
ax_schema.set_title("What the embedding lets the measure see", pad=10)

# Naive qubit: one channel (first moment = PLV).
ax_schema.text(0.02, 0.86, r"naive qubit  $\rho_{AB}$  (4$\times$4)",
               fontsize=11, fontweight="bold", color=COLORS["text"])
ax_schema.add_patch(plt.Rectangle((0.04, 0.66), 0.42, 0.12, facecolor=COLORS["quantum"],
                                  edgecolor="white", lw=1.2, alpha=0.9))
ax_schema.text(0.25, 0.72, "1st circular moment = PLV", ha="center", va="center",
               fontsize=9.5, color=COLORS["text"])

# Richer qutrit: two channels (first + second moment).
ax_schema.text(0.02, 0.46, r"richer qutrit  $\rho_{AB}$  (9$\times$9)",
               fontsize=11, fontweight="bold", color=COLORS["text"])
ax_schema.add_patch(plt.Rectangle((0.04, 0.26), 0.42, 0.12, facecolor=COLORS["quantum"],
                                  edgecolor="white", lw=1.2, alpha=0.9))
ax_schema.text(0.25, 0.32, "1st moment = PLV", ha="center", va="center",
               fontsize=9.5, color=COLORS["text"])
ax_schema.add_patch(plt.Rectangle((0.52, 0.26), 0.44, 0.12, facecolor=COLORS["highlight"],
                                  edgecolor="white", lw=1.2, alpha=0.95))
ax_schema.text(0.74, 0.32, "2nd moment (NEW)", ha="center", va="center",
               fontsize=9.5, color=COLORS["text"])
ax_schema.annotate("", xy=(0.74, 0.24), xytext=(0.74, 0.10),
                   arrowprops=dict(arrowstyle="-", color=COLORS["muted"], lw=1.0))
ax_schema.text(0.74, 0.06, "the channel PLV cannot reach", ha="center", va="center",
               fontsize=9, style="italic", color=COLORS["muted"])

fig.suptitle("The capstone result: a richer embedding opens a channel beyond PLV",
             fontsize=14, fontweight="bold", y=1.02)
fig.tight_layout()
plt.show()

**Read the figure.** Two panels telling the result from both ends.

The *left panel* stacks the three gaps `15` measured between the two engineered ensembles — ensembles that share a PLV but differ in their second circular moment. The two grey bars are the first-moment views: PLV itself and the **naive**-embedding quantum mutual information. Both sit at or below the dashed noise floor (the largest gap the matched-$a_2$ null ever produced) — they cannot tell the two systems apart, because the only thing that differs is a moment they do not read. The rose bar is the **richer**-embedding QMI: its gap stands several times above that floor. Same two systems, three measures, one verdict — only the measure with access to the second moment separates them.

The *right panel* says *why*, in the structure of $\rho_{AB}$. The naive qubit's joint density has a single coupling channel, and it carries the first circular moment — which *is* PLV (notebook `13`). The richer qutrit's joint density keeps that channel and adds a second, carrying the second circular moment — a slot with no counterpart at all in the $4\times 4$ naive density. The advantage in the left panel is this extra channel in the right one, made into a number. Note the honest scale: the winning gap is small in absolute terms — about $0.004$ nats. That smallness is not a flaw in the demonstration; it is the first entry in the accounting we turn to now.

## 2. The honest accounting

This is the heart of the synthesis. A demonstration earns trust not by being celebrated but by being *bounded* — by stating, as carefully as the result itself, where it applies, what it costs, and what it does not establish. We owe the win from `15` three things.

### When QOT helps

The advantage appears precisely **when the coupling carries higher-order, structured content that the embedding captures, beyond the first circular moment PLV reads.** In `15` that content was the second moment $a_2$; the richer qutrit had a channel for it, and so the QOT measure saw a difference PLV could not. This is conditional, and the conditions matter in both directions:

- On the **naive embedding**, QOT does *not* help — `13` proved it is PLV relabelled. The advantage is bought by the richer embedding, never by the word *quantum*.
- For coupling that is **purely first-moment** — a dyad whose phase-difference distribution differs only in how concentrated it is — there is nothing past PLV to find, and QOT equals PLV again. Higher-order machinery earns its keep only when there is higher-order structure to read.

So the honest scope is narrow and precise: QOT adds something exactly where the system's coupling has structure beyond the first moment *and* the embedding is rich enough to encode it. That is a real regime — but it is not all regimes.

### At what statistical cost

The richer embedding is not free. Three costs, all visible in what we built:

- **More degrees of freedom.** The qutrit's joint density is $9\times 9$, against the qubit's $4\times 4$. More parameters to estimate from the same data means more ways for finite-sample noise to enter — the extra channels must be estimated, not assumed.
- **The effect can be small, and the null has a tail.** The separation in `15` was about $0.004$ nats — genuine, and clear of its null across a dozen seeds, but modest in absolute size. Detecting it reliably needed $N = 150{,}000$ samples per channel so that the Monte-Carlo error (which falls like $1/\sqrt{N}$) shrank below the gap. With fewer samples the $a_2$-matched null's upper tail would reach up toward the real effect, and the clean separation we saw would blur.
- **Higher-dimensional states cost more to estimate and to solve.** A qutrit-by-qutrit state, or any further enrichment, raises the dimension of every object the transport and information measures act on — more expensive to estimate from data and more expensive to compute. The reach of a richer embedding is paid for in samples and in arithmetic.

None of this negates the result. It prices it. The advantage is real *and* it has a bill, and reading a result honestly means reading both lines.

### The caveat — do not overclaim

Here is the claim the demonstration does **not** license, stated plainly so it cannot be smuggled back in: **it is not true that only a quantum measure could detect this difference.** The two ensembles differ in their second circular moment, and a purely *classical* higher-order statistic — a second-moment (second-harmonic) phase-locking value $|\langle e^{2i\Delta\theta}\rangle|$, or the bispectrum — would also tell them apart. There is no quantum magic here: the second moment is a classical feature of the phase-difference distribution, and a classical statistic aimed at it sees it directly. To claim *only QOT can* would be to overclaim, and the course does not. And this is not merely an observation: it is a **structural bound**, demonstrated in `18_what_no_classical_data_embedding_can_buy`. Because $\rho_{AB}$ is built deterministically from the classical phases, every one of its elements is a classical circular moment — so a same-order classical statistic matches any QOT measure, and *entangling* the embedding does not change that. The boundary is precise, and it has an upside: `17_embedding_design_amplifies_the_effect` shows the same embedding-as-dial that bounds *exclusivity* can be turned to make the effect comfortably **measurable** — lifting it across the hardware noise floor by down-weighting the channel PLV already reads.

What, then, *is* the achievement? The value of QOT is **not** that it alone reaches the second moment; it is the **unified geometric framework**. Quantum optimal transport places classical and quantum coupling in one transport / information-geometric language — the classical↔quantum dictionary built across the whole course — and it accesses higher-order structure **naturally, through the embedding choice**, rather than through a statistic hand-picked in advance. With a classical approach you must *decide* to compute the second moment, then the third, then the bispectrum, each a separate bespoke choice. With QOT you choose a richer embedding and the same two measures — quantum mutual information, the Bures distance — automatically read whatever structure that embedding now carries. The advantage is of *framework*, not of magic: one geometry, one pair of measures, and the embedding as the single dial that sets what the coupling measure can see. This is the signature of a **quantum-inspired** method: the quantum formalism borrowed as a tool for *classical* data, valuable for how it unifies and generalises the analysis (and scales to networks of many nodes), not for any detection power classical statistics lack.

That is the mature reading. We demonstrated a controlled, significant case where QOT beats PLV (the left panel above is not a promise but a measurement); we priced it honestly; and we located its value exactly — in the unifying language, not in a uniqueness it does not have. Holding all of that at once is what it means to take the result seriously.

## 3. Closing the dictionary row

Every module of this course added rows to one classical↔quantum dictionary: a table in which each classical object meets its quantum counterpart in a shared transport / information-geometric language. Module 3 closed the transport rows (covariance ↔ density matrix, $W_2$ ↔ Bures, McCann ↔ quantum coupling). The capstone closes the last row — the one about **coupling measures** themselves, the row this whole arc was written to fill.

| Classical | Quantum | The bridge the capstone built |
|---|---|---|
| **Phase-locking value** $\;\mathrm{PLV} = |\langle e^{i\Delta\theta}\rangle|$ | **first-moment coherence** of $\rho_{AB}$ — $|\rho_{AB}[1,2]| = \mathrm{PLV}/4$ on the naive embedding | PLV *is* the first circular moment, read off one matrix element (`13`) |
| **higher-order phase statistics** (2nd-moment PLV, bispectrum) | **QMI / Bures-coupling** $\;d(\rho_{AB},\,\rho_A\otimes\rho_B)$ — relative-entropy / transport distance from the product | a single distance-from-independence, reading every channel the embedding carries (`14`, `15`) |
| choosing *which statistic* to compute (1st? 2nd? bispectrum?) | **the embedding choice** $\;\theta \mapsto |\psi(\theta)\rangle$ | the embedding sets *which structure the coupling measure can see* — the single dial of the framework (`10`, `13`, `14`) |

Read the rows together and the punchline of the capstone is the table itself. A classical coupling measure is a *choice of statistic*: PLV picks the first moment, a second-harmonic PLV picks the second, the bispectrum picks a particular triple interaction — each a separate hand-made decision. The quantum side replaces that menu of statistics with **one** pair of geometric quantities (the relative-entropy distance QMI and the transport distance Bures) acting on $\rho_{AB}$, and moves the *choice* upstream into the embedding. Pick the naive qubit and the quantum measure reads exactly PLV; pick the richer qutrit and the same measure also reads the second moment. The dictionary's final row is therefore not *PLV versus QMI* as rivals — it is the statement that PLV is the first-moment special case of a single geometric coupling measure whose reach is set, in one place, by how you turn the signal into a state.

## Your turn

These are reasoning and design prompts — no new code is required to think them through; they consolidate the accounting above.

**Warm-up.** In your own words, complete the sentence three ways: *"QOT does not beat PLV when ___."* Give one answer about the **embedding**, one about the **kind of coupling** in the system, and one about the **sample size**. Each should point back to a specific notebook in the arc (`13`, the structure of the coupling, or the Monte-Carlo cost in `15`).

**Go further.** A colleague reads the left panel of section 1 and says: "This proves you need quantum optimal transport to detect second-moment coupling." Write the two- or three-sentence correction you would send back. Name the classical statistic that would also separate the two ensembles, and then state what QOT genuinely contributes that the classical statistic does not — framing it as a property of the *framework*, not a uniqueness claim.

**Challenge.** Suppose you wanted to extend the demonstration so that the difference between two ensembles lived in the **third** circular moment $\langle e^{3i\Delta\theta}\rangle$, with the first two matched. Sketch, in words, the whole experiment: which embedding you would choose and why (recall the qutrit of `14` is blind past the second harmonic), which channel of the enlarged $\rho_{AB}$ would carry the distinction, what classical statistic would be the fair baseline, and what control you would run to show any separation is real rather than Monte-Carlo noise. Then say, in one sentence, what added *cost* this richer embedding would carry over the qutrit — tying it to the accounting in section 2.

## What you built

This is the close of the capstone, and of the course's central argument. You did not settle for *"the quantum machinery looks reasonable"*; you put it to a test it could fail, and read the outcome with both eyes open.

- You assembled the whole arc into one argument: the **redundancy reveal** (`13` — the naive quantum measures are PLV in disguise, for a structural reason), the **richer embedding** (`14` — the qutrit's $\rho_{AB}$ carries the second moment in a channel independent of PLV), and the **discriminating experiment** (`15` — a controlled, significant separation of two matched-PLV ensembles).
- You stated the demonstrated result **precisely and no further**: a richer-embedding QOT measure separates two ensembles PLV cannot — not that only a quantum measure could, since a classical second-moment statistic would also see the difference.
- You gave the **honest accounting** in full — *when* QOT helps (higher-order structure the embedding captures, never the naive embedding, never pure first-moment coupling), *at what cost* (more degrees of freedom, a small effect against a null with a tail, higher-dimensional states to estimate and solve), and the *caveat* (classical higher-order statistics can chase any single moment too).
- You named the genuine value: a **unified transport / information-geometric framework** that reaches higher-order coupling structure through the embedding choice rather than a hand-picked statistic — and you **closed the coupling-measure row** of the classical↔quantum dictionary, with PLV revealed as the first-moment special case of one geometric coupling measure.

Take the full measure of this. You built quantum optimal transport from the diagonal collapse all the way to a working coupling measure, you found the honest place where it earns its keep, and you held the celebration and the caveats in the same hand. That balance — a real result, precisely bounded — is what mature scientific work looks like, and it is yours now.

## What's next

The capstone's *result* is complete; its *reckoning* continues. In `17_embedding_design_amplifies_the_effect` we turn the embedding dial and make the modest effect comfortably measurable; in `18_what_no_classical_data_embedding_can_buy` we find the precise structural boundary of the quantum-inspired program (and why entangling the embedding cannot cross it). Then `19_vqe_limitation_de_palma` turns the geometry around to a frontier *proof* result — De Palma et al. bounding a whole family of quantum algorithms — before the course comes back into one frame in `20_frontier_synthesis`.

## References

- J.-P. Lachaux, E. Rodriguez, J. Martinerie & F. J. Varela, "Measuring phase synchrony in brain signals", *Human Brain Mapping* **8**, 194–208 (1999). DOI:10.1002/(SICI)1097-0193(1999)8:4<194::AID-HBM4>3.0.CO;2-C — the phase-locking value, the first-moment baseline this arc is measured against.
- K. V. Mardia & P. E. Jupp, *Directional Statistics* (Wiley, 2000). DOI:10.1002/9780470316979 — circular moments of a phase distribution; the first and second moments are distinct functionals, which is what lets QOT see past PLV and what a classical higher-order statistic could also read.
- C. Spearman, "The proof and measurement of association between two things", *American Journal of Psychology* **15**, 72–101 (1904). DOI:10.2307/1412159 — the rank correlation that exposed the redundancy in `13`.
- M. M. Wilde, *Quantum Information Theory*, 2nd ed. (Cambridge University Press, 2017). DOI:10.1017/9781316809976 — quantum mutual information $I(A{:}B) = S(\rho_A) + S(\rho_B) - S(\rho_{AB})$, one of the two geometric coupling measures.
- D. Trevisan, "Optimal transport methods for quantum systems", arXiv:2202.02091 (2022). DOI:10.48550/arXiv.2202.02091 — the quantum-transport coupling measures (quantum mutual information, Bures) that read the enriched $\rho_{AB}$.
- G. De Palma, M. Marvian, D. Trevisan & S. Lloyd, "The quantum Wasserstein distance of order 1", *IEEE Trans. Inf. Theory* **67**, 6627–6643 (2021). DOI:10.1109/TIT.2021.3076442 — the qubit $W_1$ formulation behind the frontier notebook to come.

**Previous:** `notebooks/04_QuantumOptimalTransport/15_discriminating_experiment.ipynb` · **Next:** `notebooks/04_QuantumOptimalTransport/17_embedding_design_amplifies_the_effect.ipynb`